# Zulassungskarten

analysis of the results from pipeline.py

In [1]:
# imports
import json
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path

# Load & prepare Data

## Load

In [2]:
DATA_DIR = Path("../../data/03_processed") 

records = []

for file_path in DATA_DIR.glob("*.json"):
    with file_path.open(encoding="utf-8") as file:
        record = json.load(file)

    record["_file_name"] = file_path.name
    records.append(record)

df_processed = pd.json_normalize(records, sep="_")
df = df_processed.copy()

# df

## Helpers Functions

In [3]:
def parse_meters(value):
    if pd.isna(value):
        return float("nan")

    text = str(value).lower().replace("m", "").strip()
    if not text:
        return float("nan")

    # German decimal comma and thousands separators
    if "," in text:
        text = text.replace(".", "").replace(",", ".")
    else:
        text = text.replace(".", "")

    return pd.to_numeric(text, errors="coerce")

In [4]:
MONTH_NUMBERS = {
    "januar": 1,
    "februar": 2,
    "märz": 3,
    "april": 4,
    "mai": 5,
    "juni": 6,
    "juli": 7,
    "august": 8,
    "september": 9,
    "oktober": 10,
    "november": 11,
    "dezember": 12,
}


def parse_german_date(value):
    if pd.isna(value):
        return pd.NaT

    parts = str(value).strip().rstrip(".").split()
    if len(parts) != 3:
        return pd.NaT

    day = parts[0].rstrip(".")
    month = MONTH_NUMBERS.get(parts[1].lower())
    year = parts[2]

    if month is None:
        return pd.NaT

    return pd.to_datetime(
        f"{year}-{month:02d}-{day}",
        errors="coerce",
    )

## Prepare data

- join metadata with processed
- metadata columns are prefixed with `metadata_`; unprefixed data columns come from processed
- `metadata_available` marks rows with a matching metadata file
- sort by date or examinationnumber?

In [5]:
METADATA_DIR = Path("../../data/01_raw")

if "df_processed" not in globals():
    df_processed = pd.json_normalize(records, sep="_")

metadata_records = []

for file_path in METADATA_DIR.rglob("*.json"):
    with file_path.open(encoding="utf-8") as file:
        metadata_record = json.load(file)

    # Match the metadata file to its corresponding processed file.
    metadata_record["_file_name"] = file_path.name.replace(".json", "_processed.json")
    metadata_records.append(metadata_record)

df_metadata = pd.json_normalize(metadata_records, sep="_")
df_metadata = df_metadata.rename(
    columns={
        column: f"metadata_{column}"
        for column in df_metadata.columns
        if column != "_file_name"
    }
)

df = df_processed.merge(
    df_metadata,
    on="_file_name",
    how="left",
    validate="one_to_one",
    indicator="_metadata_join",
)
df["metadata_available"] = df.pop("_metadata_join").eq("both")

# df

In [6]:
df["examination_number_parsed"] = pd.to_numeric(df["examination_number"], errors="coerce").astype("Int64")
df["metadata_Prüfnummer_parsed"] = pd.to_numeric(df["metadata_Prüfnummer"], errors="coerce").astype("Int64")

df["metadata_Länge_parsed"] = df["metadata_Länge (in Meter)"].apply(parse_meters)
#  df["length_details_individual_acts"] ???
df["total_original_length_parsed"] = df["length_details_total_length_original_length"].apply(parse_meters)
df["total_length_after_cut_parsed"] = df["length_details_total_length_length_after_cut"].apply(parse_meters)

df["examination_certificate_date_parsed"] = df["examination_certificate_date"].apply(parse_german_date) # nicht jedes wird geparsed aufgrund von schlechter ocr erkennung hmmm
df["metadata_Prüfdatum_parsed"] = pd.to_datetime(df["metadata_Prüfdatum"], dayfirst=True, errors="coerce") # hier auch extra pasred spalte?
df["metadata_year"] = df["metadata_Prüfdatum_parsed"].dt.year.astype("Int64")

In [7]:
with pd.option_context("display.max_columns", None):
    display(df)

,examination_number,stamp_text,origin_company,movie_title,movie_subtitle,produced_by,director,cinematography,set_design,cast_list,excerpt_note,intertitles,_file_name,length_details_individual_acts,length_details_total_length_original_length,length_details_total_length_length_after_cut,examination_certificate_examination_board,examination_certificate_location,examination_certificate_date,examination_certificate_decision_text,metadata_Titel_und_Signatur,metadata_Prüfnummer,metadata_Prüfdatum,metadata_Antragsteller,metadata_Produktion,metadata_Land,metadata_Filmart,metadata_Stumm-/Tonfilm,metadata_Länge (in Meter),metadata_Entscheidung,metadata_Unterlagenart,metadata_Benutzungsort,metadata_Bemerkung,metadata_Vorführungsbeschränkung,metadata_Ort des Prüfdatums,metadata_available,examination_number_parsed,metadata_Prüfnummer_parsed,metadata_Länge_parsed,total_original_length_parsed,total_length_after_cut_parsed,examination_certificate_date_parsed,metadata_Prüfdatum_parsed,metadata_year
0,11751,Zulassungskarten für Bildstreifen sind öffentl...,"Universal Pictures Corp., New York",Liebestoll.,NaN,Deutsch-Nordische Film-Union G. m. b. H.,NaN,NaN,NaN,[],NaN,"[{'act_label': '1. Akt', 'text': '1. Akt. 1. E...",R 9346-I_7746_processed.json,"[{'act_number': 'I', 'original_length': '260 m...",498 m,NaN,Film-Prüfstelle Berlin,Berlin,13. November 1935,Der Bildstreifen wird zur öffentlichen Vorführ...,R 9346-I/7746\nLiebestoll.\n1920 - 1945,11751,13.11.1925,"Deutsch-Nordische Film-Union GmbH, Berlin","Universal-Pictures Corp., New York",USA,Spielfilm,Stummfilm,498,Jugendverbot,Zulassungskarte,Berlin-Lichterfelde,NaN,NaN,NaN,True,11751,11751,498.0,498.0,NaN,1935-11-13,1925-11-13,1925
1,38954,Zulassungskarten für Filme sind öffentliche Ur...,"Gebrüder Diehl-Film, Gräfeling bei München",Die Macht der Liebe.,Tobis zeigt einen Puppenfilm der Gebrüder Dieh...,Killerstraße 10/1.,NaN,Hans Asen,NaN,[],NaN,"[{'act_label': 'Fortsetzung', 'text': '2. Aha,...",R 9346-I_24154_processed.json,[],580 m,NaN,Film-Prüfstelle,Berlin,29. März 1935,Der Film wird zur öffentlichen Vorführung im D...,R 9346-I/24154\nDie Macht der Liebe\n1920 - 1945,38954,29.3.1935,"Gebr. Diehl, Gräfelfing b. München","Gebr. Diehl, Gräfelfing b. München",Deutschland,Spielfilm,Tonfilm,580,Jugendfrei,Zulassungskarte,Berlin-Lichterfelde,NaN,NaN,NaN,True,38954,38954,580.0,580.0,NaN,1935-03-29,1935-03-29,1935
2,16811,Zulassungskarten für Bildstreifen sind öffentl...,"Paramount-Film, U. S. A.",Ko Ko als Wedder.,Alfred Weiß zeigt Tintenmännchen im: Ko Ko als...,Universum-Film Aktiengesellschaft Berlin SW 68...,NaN,NaN,NaN,[],NaN,[],R 9346-I_11484_processed.json,[],181 m,NaN,Film-Prüfstelle Berlin,Berlin,3. Oktober 1927,Der Bildstreifen wird zur öffentlichen Vorführ...,R 9346-I/11484\nKo Ko als Wecker.\n1920 - 1945,16811,3.10.1927,"Universum-Film AG, Berlin","Paramount, New York",USA,Zeichentrickfilm,Stummfilm,181,Jugendfrei,Zulassungskarte,Berlin-Lichterfelde,NaN,NaN,NaN,True,16811,16811,181.0,181.0,NaN,1927-10-03,1927-10-03,1927
3,57539,NaN,"Tobis Filmkunst G. m. b. H., Berlin NW 7",Die Entlassung.,Tobis zeigt Emil Jannings in: Die Entlassung.,Friedrichstraße 100,Wolfgang Liebeneiner,Fritzz Wagner,Otto Hunte,"[{'role': 'Fürst Bismarck', 'actor': 'Emil Jan...",NaN,"[{'act_label': '1. Akt', 'text': '1. Vor dem A...",R 9346-I_35012_processed.json,"[{'act_number': '1. Rolle 1. Akt', 'original_l...",2991 m,NaN,Filmprüfstelle,Berlin,28. August 1942,Der Film wird zur öffentlichen Vorführung im D...,R 9346-I/35012\nDie Entlassung\n1920 - 1945,57539,28.8.1942,"Tobis Filmkunst GmbH, Berlin","Tobis Filmkunst GmbH, Berlin",Deutschland,Spielfilm,Tonfilm,2991,Jugendfrei vom 14. Lebensjahr ab,Zulassungskarte,Berlin-Lichterfelde,NaN,NaN,NaN,True,57539,57539,2991.0,2991.0,NaN,1942-08-28,1942-08-28,1942
4,15543,Zulassungskarten für Bildstreifen sind öffent....,"Berlin W 35, Genhinter Straße 32",Die Entstehung der Pelzmode.,Pinschewer-Film. 1. Frau Eva spricht zu ihrem ...,"Werbefilm G. m. b. H., Leitung J. Pin

## Evaluation

In [8]:
def compare_ocr_with_metadata(ocr_values, metadata_values):
    # Vergleicht die beiden Spalten zeilenweise.
    # Fehlende OCR-Werte sind bei vorhandenem Ground Truth-Wert falsch.
    # Fehlen die Metadaten, ist der Fall ohne weitere Prüfung unbekannt.
    ocr_values = pd.Series(ocr_values)
    metadata_values = pd.Series(metadata_values)

    if len(ocr_values) != len(metadata_values):
        raise ValueError("Die beiden Spalten müssen gleich lang sein.")

    original_index = ocr_values.index
    ocr_values = ocr_values.reset_index(drop=True)
    metadata_values = metadata_values.reset_index(drop=True)

    metadata_available = metadata_values.notna()
    both_values_present = metadata_available & ocr_values.notna()

    identical = pd.Series(pd.NA, index=ocr_values.index, dtype="boolean")
    identical.loc[metadata_available] = False
    identical.loc[both_values_present] = ocr_values.loc[both_values_present].eq(
        metadata_values.loc[both_values_present]
    )
    evaluated_count = int(metadata_available.sum())
    correct_count = int(identical.loc[metadata_available].sum())
    accuracy = correct_count / evaluated_count if evaluated_count else None
    accuracy_text = f"{accuracy:.2%}" if accuracy is not None else "n/a"
    unverified_count = int((~metadata_available).sum())
    ocr_without_metadata_count = int(
        (~metadata_available & ocr_values.notna()).sum()
    )

    print(
        f"{correct_count} von {evaluated_count} bewertbaren Zeilen korrekt "
        f"({accuracy_text})"
    )
    print(
        f"{unverified_count} Zeilen nicht bewertbar; davon "
        f"{ocr_without_metadata_count} mit OCR-Wert ohne Metadaten"
    )
    identical.index = original_index
    return identical


In [9]:
# Titel und Signatur
def parse_title(df, column_name):
    return df[column_name].str.split('\n').str[1] 

df["metadata_Titel"] = parse_title(df, "metadata_Titel_und_Signatur")
df["movie_title_parsed"] = df["movie_title"].str.replace('.', '', regex=False)

identical_titel = compare_ocr_with_metadata(
    df["movie_title"],
    df["metadata_Titel"]
) 

print("\n")

identical_titel = compare_ocr_with_metadata(
    df["movie_title_parsed"],
    df["metadata_Titel"]
)

df_look = df[df["metadata_Titel"].isna()]
# df_look


with pd.option_context("display.max_columns", None):
    display(df_look)

472 von 2823 bewertbaren Zeilen korrekt (16.72%)
1 Zeilen nicht bewertbar; davon 1 mit OCR-Wert ohne Metadaten


745 von 2823 bewertbaren Zeilen korrekt (26.39%)
1 Zeilen nicht bewertbar; davon 1 mit OCR-Wert ohne Metadaten


,examination_number,stamp_text,origin_company,movie_title,movie_subtitle,produced_by,director,cinematography,set_design,cast_list,excerpt_note,intertitles,_file_name,length_details_individual_acts,length_details_total_length_original_length,length_details_total_length_length_after_cut,examination_certificate_examination_board,examination_certificate_location,examination_certificate_date,examination_certificate_decision_text,metadata_Titel_und_Signatur,metadata_Prüfnummer,metadata_Prüfdatum,metadata_Antragsteller,metadata_Produktion,metadata_Land,metadata_Filmart,metadata_Stumm-/Tonfilm,metadata_Länge (in Meter),metadata_Entscheidung,metadata_Unterlagenart,metadata_Benutzungsort,metadata_Bemerkung,metadata_Vorführungsbeschränkung,metadata_Ort des Prüfdatums,metadata_available,examination_number_parsed,metadata_Prüfnummer_parsed,metadata_Länge_parsed,total_original_length_parsed,total_length_after_cut_parsed,examination_certificate_date_parsed,metadata_Prüfdatum_parsed,metadata_year,metadata_Titel,movie_title_parsed
1450,54802,Zulassungskarten für Filme sind öffentliche Ur...,"Gasparcolor Werbefilme G. m. b. H., Berlin W 50",Der Imbert-Generator.,"Ein Film der Gasparcolor-Film, Berlin W 50.","Paul Schwärzel,\nunter Mitarbeit von Hasso Pre...",NaN,NaN,NaN,[],NaN,[],R_9346_I_33595_processed.json,[],NaN,NaN,Film-Prüfstelle,Berlin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,54802,<NA>,NaN,NaN,NaN,NaT,NaT,<NA>,NaN,Der Imbert-Generator


In [10]:
identical_examination_number = compare_ocr_with_metadata(
    df["examination_number"],
    df["metadata_Prüfnummer"],
)

# Manuell geprüfte Fälle ohne Metadaten hier eintragen.
manual_evaluation = {
    # "R 9346-I_123_processed.json": True,
    # "R 9346-I_456_processed.json": False,
}

manual_correct = df["_file_name"].map(manual_evaluation).astype("boolean")
evaluation = identical_examination_number.fillna(manual_correct)

evaluated_count = int(evaluation.notna().sum())
correct_count = int(evaluation.eq(True).sum())
accuracy = correct_count / evaluated_count if evaluated_count else None
accuracy_text = f"{accuracy:.2%}" if accuracy is not None else "n/a"
print(
    f"Nach manueller Bewertung: {correct_count} von {evaluated_count} "
    f"Zeilen korrekt ({accuracy_text})"
)

errors = evaluation.eq(False)

columns = [
    "examination_number",
    "metadata_Prüfnummer",
    "_file_name"
]
df_errors = df.loc[errors.fillna(False)][columns]
df_errors

2670 von 2799 bewertbaren Zeilen korrekt (95.39%)
25 Zeilen nicht bewertbar; davon 25 mit OCR-Wert ohne Metadaten
Nach manueller Bewertung: 2670 von 2799 Zeilen korrekt (95.39%)


,examination_number,metadata_Prüfnummer,_file_name
27,1999,4999,R 9346-I_2715_processed.json
31,1666,4666,R 9346-I_2459_processed.json
41,8378 / 122,8378,R 9346-I_5194_processed.json
87,4098,4093,R 9346-I_2031_processed.json
124,8607\n8608,8607,R 9346-I_5366_processed.json
...,...,...,...
2708,3901,3801,R 9346-I_1854_processed.json
2752,168 7462,7468,R 9346-I_4557_processed.json
2754,441 40,44140,R 9346-I_27057_processed.json
2785,601,661,R 9346-I_120_processed.json


In [11]:
# prüfdatum
identical_date = compare_ocr_with_metadata(
    df["examination_certificate_date_parsed"],
    df["metadata_Prüfdatum_parsed"]
)

2428 von 2823 bewertbaren Zeilen korrekt (86.01%)
1 Zeilen nicht bewertbar; davon 0 mit OCR-Wert ohne Metadaten


In [12]:
df_look = df[df["examination_certificate_date_parsed"].isna()]

df_look = df_look[["_file_name","examination_certificate_date_parsed", "examination_certificate_date"]]

df_look

,_file_name,examination_certificate_date_parsed,examination_certificate_date
53,R 9346-I_23157_processed.json,NaT,16 December 1933
164,R 9346-I_34446_processed.json,NaT,NaN
198,R 9346-I_22946_processed.json,NaT,NaN
300,R 9346-I_8062_processed.json,NaT,19. Januar 1926; 03. September 1935
368,R 9346-I_21023_processed.json,NaT,20. August 1932 / 27. Juni 1932
418,R 9346-I_3947_processed.json,NaT,NaN
456,R 9346-I_13978_processed.json,NaT,31. September 1928
492,R 9346-I_19316_processed.json,NaT,8. April 1931 / 8. Januar 1936
495,R 9346-I_19290_processed.json,NaT,1. April 1931; 3. Oktober 1935
513,R 9346-I_1862_processed.json,NaT,NaN


In [13]:
identical_wtf = compare_ocr_with_metadata(
    df["metadata_Antragsteller"],
    df["metadata_Produktion"]
)

2123 von 2819 bewertbaren Zeilen korrekt (75.31%)
5 Zeilen nicht bewertbar; davon 1 mit OCR-Wert ohne Metadaten


In [14]:
# antragsteller & produktion
identical_origin_company = compare_ocr_with_metadata(
    df["origin_company"],
    df["metadata_Antragsteller"]
)

print("\n")

identical_origin_company_2 = compare_ocr_with_metadata(
    df["origin_company"],
    df["metadata_Produktion"]
)

print("\n")

identical_produced_by = compare_ocr_with_metadata(
    df["produced_by"],
    df["metadata_Antragsteller"]
)

print("\n")

identical_produced_by = compare_ocr_with_metadata(
    df["produced_by"],
    df["metadata_Produktion"]
)

df_look = df[["origin_company", "metadata_Antragsteller"]]
df_look

23 von 2820 bewertbaren Zeilen korrekt (0.82%)
4 Zeilen nicht bewertbar; davon 4 mit OCR-Wert ohne Metadaten


50 von 2819 bewertbaren Zeilen korrekt (1.77%)
5 Zeilen nicht bewertbar; davon 5 mit OCR-Wert ohne Metadaten


17 von 2820 bewertbaren Zeilen korrekt (0.60%)
4 Zeilen nicht bewertbar; davon 4 mit OCR-Wert ohne Metadaten


38 von 2819 bewertbaren Zeilen korrekt (1.35%)
5 Zeilen nicht bewertbar; davon 5 mit OCR-Wert ohne Metadaten


,origin_company,metadata_Antragsteller
0,"Universal Pictures Corp., New York","Deutsch-Nordische Film-Union GmbH, Berlin"
1,"Gebrüder Diehl-Film, Gräfeling bei München","Gebr. Diehl, Gräfelfing b. München"
2,"Paramount-Film, U. S. A.","Universum-Film AG, Berlin"
3,"Tobis Filmkunst G. m. b. H., Berlin NW 7","Tobis Filmkunst GmbH, Berlin"
4,"Berlin W 35, Genhinter Straße 32","Werbefilm GmbH, Berlin"
...,...,...
2819,"Paramount Film A.-G., Berlin SW 68 Friedrichst...","Paramount-Film AG, Berlin"
2820,"C. A. Linke & Co., Dresden, Jüdenhof 2","C.A. Linke u. Co., Dresden"
2821,"Fox-Film Corporation, New York, Amerika",Deulig-Film
2822,"Briesse Schmalfilm, Dr. Gerd Briese","Briese-Schmalfilm, Dr. Gerd Briese, Berlin-Cha..."


In [16]:
# Länge (in Meter)

identical_length = compare_ocr_with_metadata(
    df["total_original_length_parsed"],
    df["metadata_Länge_parsed"]
)

2343 von 2778 bewertbaren Zeilen korrekt (84.34%)
46 Zeilen nicht bewertbar; davon 42 mit OCR-Wert ohne Metadaten


# Analysis

## Filmlength & cuts

In [ ]:
df_filmlength = df[[
    "_file_name", 
    "movie_title",
    "metadata_year",
    "metadata_Länge_parsed", 
    "total_original_length_parsed", 
    "total_length_after_cut_parsed", 
    "length_details_individual_acts"
]]

df_filmlength["cut_length_available"] = (
    df_filmlength["total_original_length_parsed"].notna()
    & df_filmlength["total_length_after_cut_parsed"].notna()
)

valid_lengths = (
    df_filmlength["cut_length_available"]
    & df_filmlength["total_original_length_parsed"].gt(0)
    & df_filmlength["total_length_after_cut_parsed"].le(df_filmlength["total_original_length_parsed"])
)

df_filmlength["cut_meters"] = (
    df_filmlength["total_original_length_parsed"] - df_filmlength["total_length_after_cut_parsed"]
).where(valid_lengths)

df_filmlength["cut_rate"] = (
    df_filmlength["cut_meters"] / df_filmlength["total_original_length_parsed"]
).where(valid_lengths)

# yearly_length["cut_share_percent"] = (
#     yearly_length["films_with_cut"]
#     .div(yearly_length["records_with_cut_length"])
#     .mul(100)
#     .where(yearly_length["records_with_cut_length"].gt(0))
# )

df_filmlength

,_file_name,movie_title,metadata_year,metadata_Länge_parsed,total_original_length_parsed,total_length_after_cut_parsed,length_details_individual_acts,cut_length_available,cut_meters,cut_rate
0,R 9346-I_7746_processed.json,Liebestoll.,1925,498.0,498.0,NaN,"[{'act_number': 'I', 'original_length': '260 m...",False,NaN,NaN
1,R 9346-I_24154_processed.json,Die Macht der Liebe.,1935,580.0,580.0,NaN,[],False,NaN,NaN
2,R 9346-I_11484_processed.json,Ko Ko als Wedder.,1927,181.0,181.0,NaN,[],False,NaN,NaN
3,R 9346-I_35012_processed.json,Die Entlassung.,1942,2991.0,2991.0,NaN,"[{'act_number': '1. Rolle 1. Akt', 'original_l...",False,NaN,NaN
4,R 9346-I_10472_processed.json,Die Entstehung der Pelzmode.,1927,94.0,94.0,NaN,[],False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2819,R 9346-I_20179_processed.json,Mann über Bord.,1931,2200.0,2200.0,NaN,"[{'act_number': 'I', 'original_length': '292 m...",False,NaN,NaN
2820,R 9346-I_5875_processed.json,Gehöhrte Räuber.,1924,393.0,393.0,NaN,[],False,NaN,NaN
2821,R 9346-I_4696_processed.json,Die Wölfin.,1923,539.0,539.0,NaN,"[{'act_number': 'I', 'original_length': '241 m...",False,NaN,NaN
2822,R 9346-I_29422_processed.json,Neue Möbel im neuen Heim. (Schmalfilm.),1938,16.0,16.0,NaN,[],False,NaN,NaN


In [45]:
yearly_length = (
    df_filmlength.dropna(subset=["metadata_year"])
    .groupby("metadata_year")
    .agg(
        records=("_file_name", "count"),
        recors_with_original_length=("total_original_length_parsed", "count"),
        records_with_cut_length=("cut_length_available", "sum"),
        records_length=("total_original_length_parsed", "sum"),
        records_cut_meters=("cut_meters", "sum"),

        metadata_Länge_parsed_mean=("metadata_Länge_parsed", "mean"),
        metadata_Länge_parsed_median=("metadata_Länge_parsed", "median"),

        total_original_length_mean=("total_original_length_parsed", "mean"),
        total_original_length_median=("total_original_length_parsed", "median"),

        total_length_after_cut_mean=("total_length_after_cut_parsed", "mean"),
        total_length_after_cut_median=("total_length_after_cut_parsed", "median"),

        cut_meters_mean=("cut_meters", "mean"),
        cut_meters_median=("cut_meters", "median"),
        cut_rate_mean=("cut_rate", "mean"),
        cut_rate_median=("cut_rate", "median")
    )
)



yearly_length.round(2)

KeyError: 'films_with_cut'

### Individual Acts

In [19]:
df_filmlength_acts = (
    df_filmlength
    .explode("length_details_individual_acts", ignore_index=True)
    .dropna(subset=["length_details_individual_acts"])
    .reset_index(drop=True)
)

df_filmlength_acts = pd.concat(
    [
        df_filmlength_acts.drop(columns="length_details_individual_acts"),
        pd.json_normalize(df_filmlength_acts["length_details_individual_acts"]),
    ],
    axis=1,
)

df_filmlength_acts["original_length"] = df_filmlength_acts["original_length"].apply(parse_meters)
df_filmlength_acts["length_after_cut"] = df_filmlength_acts["length_after_cut"].apply(parse_meters)  

df_filmlength_acts

,_file_name,movie_title,metadata_year,metadata_Länge_parsed,total_original_length_parsed,total_length_after_cut_parsed,act_number,original_length,length_after_cut
0,R 9346-I_7746_processed.json,Liebestoll.,1925,498.0,498.0,NaN,I,260.0,NaN
1,R 9346-I_7746_processed.json,Liebestoll.,1925,498.0,498.0,NaN,II,238.0,NaN
2,R 9346-I_35012_processed.json,Die Entlassung.,1942,2991.0,2991.0,NaN,1. Rolle 1. Akt,334.0,NaN
3,R 9346-I_35012_processed.json,Die Entlassung.,1942,2991.0,2991.0,NaN,2. Rolle 2. u. 3. Akt,536.0,NaN
4,R 9346-I_35012_processed.json,Die Entlassung.,1942,2991.0,2991.0,NaN,3. Rolle 4. u. 5. Akt,559.0,NaN
...,...,...,...,...,...,...,...,...,...
5252,R 9346-I_20179_processed.json,Mann über Bord.,1931,2200.0,2200.0,NaN,VIII,270.0,NaN
5253,R 9346-I_4696_processed.json,Die Wölfin.,1923,539.0,539.0,NaN,I,241.0,NaN
5254,R 9346-I_4696_processed.json,Die Wölfin.,1923,539.0,539.0,NaN,II,298.0,NaN
5255,R 9346-I_3928_processed.json,Eine Nacht gelebt im,1922,564.0,564.0,NaN,I,330.0,NaN


In [ ]:
yearly_source = df_filmlength.copy()
yearly_source["original_length_m"] = yearly_source["total_original_length_parsed"]
yearly_source["cut_length_m"] = yearly_source["total_length_after_cut_parsed"]

yearly_source["cut_length_available"] = (
    yearly_source["original_length_m"].notna()
    & yearly_source["cut_length_m"].notna()
)

valid_yearly_lengths = (
    yearly_source["cut_length_available"]
    & yearly_source["original_length_m"].gt(0)
    & yearly_source["cut_length_m"].le(yearly_source["original_length_m"])
)

yearly_source["cut_meters"] = (
    yearly_source["original_length_m"] - yearly_source["cut_length_m"]
).where(valid_yearly_lengths)

yearly_source["cut_rate"] = (
    yearly_source["cut_meters"] / yearly_source["original_length_m"]
).where(valid_yearly_lengths)

yearly_length = (
    yearly_source.dropna(subset=["metadata_year"])
    .groupby("metadata_year")
    .agg(
        # records=("_file_name", "count"),
        # records_with_original_length=("original_length_m", "count"),
        # records_with_cut_length=("cut_length_available", "sum"),
        films_with_cut=("cut_meters", lambda values: values.gt(0).sum()),
        # metadata_Länge_parsed_mean=("metadata_Länge_parsed", "mean"),
        # metadata_Länge_parsed_median=("metadata_Länge_parsed", "median"),
        # original_length_mean_m=("original_length_m", "mean"),
        # original_length_median_m=("original_length_m", "median"),
        # total_length_after_cut_mean=("total_length_after_cut_parsed", "mean"),
        # total_length_after_cut_median=("total_length_after_cut_parsed", "median"),
        cut_meters_mean=("cut_meters", "mean"),
        cut_meters_median=("cut_meters", "median"),
        cut_rate_mean=("cut_rate", "mean"),
        cut_rate_median=("cut_rate", "median"),
    )
)

yearly_length["cut_share_percent"] = (
    yearly_length["films_with_cut"]
    .div(yearly_length["records_with_cut_length"])
    .mul(100)
    .where(yearly_length["records_with_cut_length"].gt(0))
)

yearly_length.round(2)

,records,records_with_original_length,records_with_cut_length,films_with_cut,metadata_Länge_parsed_mean,metadata_Länge_parsed_median,original_length_mean_m,original_length_median_m,total_length_after_cut_mean,total_length_after_cut_median,cut_meters_mean,cut_meters_median,cut_rate_mean,cut_rate_median,cut_share_percent
metadata_year,,,,,,,,,,,,,,,
1917,3,3,0,0,277.33,371.0,277.33,371.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1920,71,71,10,10,1258.80,1315.0,1255.42,1315.0,2011.75,2043.00,19.25,12.50,0.01,0.01,100.00
1921,120,116,17,14,1035.22,998.5,1029.57,998.5,1592.55,1459.15,15.67,3.98,0.01,0.00,82.35
1922,120,116,3,3,767.97,387.5,769.96,385.0,1992.92,2159.50,14.75,17.25,0.01,0.01,100.00
1923,120,118,8,8,931.75,555.0,932.82,552.5,1751.84,1827.65,9.91,4.90,0.01,0.00,100.00
1924,120,118,5,5,964.28,629.5,965.64,636.0,1788.70,1756.10,20.70,12.05,0.01,0.01,100.00
1925,120,118,0,0,757.47,470.0,763.93,477.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1926,120,117,6,5,708.90,270.0,732.09,308.0,1559.88,2026.48,21.54,5.25,0.02,0.01,83.33
1927,120,119,4,4,661.33,244.0,663.97,250.0,1700.82,2027.55,3.68,3.45,0.00,0.00,100.00


In [ ]:
df_filmlength = df[["_file_name", "movie_title", "length_details_individual_acts", "metadata_Länge_parsed", "total_original_length_parsed", "total_length_after_cut_parsed", "metadata_year"]].copy()
df_filmlength["original_length_m"] = df_filmlength["total_original_length_parsed"]
df_filmlength["cut_length_m"] = df_filmlength["total_length_after_cut_parsed"]
# auch hier individual acts angucken??
# ohne cuts haben keine cuts

df_filmlength["cut_length_available"] = (
    df_filmlength["original_length_m"].notna()
    & df_filmlength["cut_length_m"].notna()
)

valid_lengths = (
    df_filmlength["cut_length_available"] # beide längen enthalten
    & df_filmlength["original_length_m"].gt(0) # länge größer 0
    & df_filmlength["cut_length_m"].le(df_filmlength["original_length_m"]) # gekürtze känge ist kleiner oder gleich
)

df_filmlength["cut_meters"] = (
    df_filmlength["original_length_m"] - df_filmlength["cut_length_m"]
).where(valid_lengths)

df_filmlength["cut_rate"] = (
    df_filmlength["cut_meters"] / df_filmlength["original_length_m"]
).where(valid_lengths)

# filtert die zeilen direkt heraus
columns = [
    "_file_name",
    "movie_title",
    "original_length_m",
    "cut_length_m",
    "cut_meters",
    "cut_rate",
]
df_test = df_filmlength[df_filmlength["cut_length_m"].notna()][columns]

df_test

,_file_name,movie_title,original_length_m,cut_length_m,cut_meters,cut_rate
73,R 9346-I_1216_processed.json,Verlorene Einsichten.,1313.0,1299.00,14.00,0.010663
76,R 9346-I_2688_processed.json,Freie Bahn dem Künstler.,1098.0,1093.50,4.50,0.004098
111,R 9346-I_29488_processed.json,Der Kampf mit dem Moor. (Schmalfilm.),144.0,141.35,2.65,0.018403
150,R 9346-I_16150_processed.json,Mancher Irrsinn ist.,526.0,523.80,2.20,0.004183
189,R 9346-I_31128_processed.json,Zu den Güten der Sektion Schwaben des Deutsche...,441.0,449.00,NaN,NaN
...,...,...,...,...,...,...
2700,R 9346-I_22_processed.json,Die Banditen von Asnières.,2001.0,1956.00,45.00,0.022489
2720,R 9346-I_8325_processed.json,Bat und Patchadon am See.,2186.0,2168.70,17.30,0.007914
2725,R 9346-I_10585_processed.json,Die von der Straße leben. (Allegitim.),1949.0,1944.00,5.00,0.002565
2733,R 9346-I_24976_processed.json,Miva. -- Das Vermächtnis eines Missionars.,862.0,857.50,4.50,0.005220


### jährlich?

- 10494 ist die Karte mit dem Jahr 1972

In [ ]:
# yearly_length = (
#     df_filmlength.dropna(subset=["year"])
#     .groupby("year")
#     .agg(
#         records=("_file_name", "count"),
#         records_with_original_length=("original_length_m", "count"),
#         records_with_cut_length=("cut_length_available", "sum"),
#         films_with_cut=("has_cut", "sum"),
#         original_length_mean_m=("original_length_m", "mean"),
#         original_length_median_m=("original_length_m", "median"),
#         cut_meters_mean=("cut_meters", "mean"),
#         cut_meters_median=("cut_meters", "median"),
#         cut_rate_mean=("cut_rate", "mean"),
#         cut_rate_median=("cut_rate", "median"),
#     )
# )

# yearly_length["cut_share_percent"] = (
#     yearly_length["films_with_cut"]
#     .div(yearly_length["records_with_cut_length"])
#     .mul(100)
#     .where(yearly_length["records_with_cut_length"].gt(0))
# )

# yearly_length.round(2)


# hier nicht die raus filtern die keine cut length haben ig

plot

In [ ]:
plot_data = yearly_length[yearly_length["records_with_cut_length"] > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    yearly_length.index,
    yearly_length["original_length_median_m"],
    marker="o",
    label="Median der Originallänge",
)
axes[0].set_title("Filmlänge pro Prüfjahr")
axes[0].set_xlabel("Prüfjahr")
axes[0].set_ylabel("Meter")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(
    plot_data.index,
    plot_data["cut_share_percent"],
    marker="o",
    color="firebrick",
)
axes[1].set_title("Anteil gekürzter Filme (vergleichbare Datensätze)")
axes[1].set_xlabel("Prüfjahr")
axes[1].set_ylabel("Anteil in Prozent")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## next thing ig

## Intertitles

In [ ]:
df_intertitles = pd.json_normalize(
    records,
    record_path="intertitles",
    meta=[
        "examination_number",
        "movie_title",
        "_file_name"
    ],
    errors="ignore",
    sep="_"
)

df_intertitles

,act_label,text,examination_number,movie_title,_file_name
0,Fortsetzung,gewerbes. d) Sängerurnde des Stuttgarter Falto...,14892,"Die Linotype-Setzmaschine.\nGeschichte, Fabrik...",R 9346-I_10000_processed.json
1,1. Akt,"1. Akt.\n1. Will Gallagher, ein junger verwege...",14893,"Fred, der Geßürückte.",R 9346-I_10001_processed.json
2,1. Akt,gewandt werden. 8. Uns fehlen ein paar mutige ...,14893,"Fred, der Geßürückte.",R 9346-I_10001_processed.json
3,2. Akt,1. Mein lieber Silberhörn ... jetzt fluttern w...,14893,"Fred, der Geßürückte.",R 9346-I_10001_processed.json
4,Fortsetzung,lehrers. 10. Wozu brauchen wir einen Wunderheh...,14893,"Fred, der Geßürückte.",R 9346-I_10001_processed.json
...,...,...,...,...,...
6128,3. Akt,"1. Du Bestie in Menschengestalt, Du Wölfin!!! ...",2,Die Wölfin.,R 9346-I_1_processed.json
6129,Fortsetzung,"nicht auch, Florence? 5. Ich fahre jetzt zur B...",2,Die Wölfin.,R 9346-I_1_processed.json
6130,3. Akt,"3. Akt. 1. Henry, Du versprachst mir, dich nic...",2,Die Wölfin.,R 9346-I_1_processed.json
6131,Fortsetzung,"Sprich die Wahrheit, wer hat mich ins Irrenhau...",2,Die Wölfin.,R 9346-I_1_processed.json
